# Speculative Decoding

**Stage 5: Inference Optimization - Notebook 56**

This notebook explores speculative decoding, a technique that can achieve 2-3x speedup in LLM inference without quality loss. We'll cover:
- The autoregressive bottleneck
- Speculative decoding concept (2023)
- Draft model + verification paradigm
- Speedup analysis and theoretical bounds
- Implementation from scratch
- Choosing draft models
- Quality preservation guarantees
- Production deployment strategies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Optional, Callable
import time
from dataclasses import dataclass
from collections import defaultdict

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. The Autoregressive Bottleneck

### Historical Context

**The Problem**:
- Transformer generation is autoregressive: must generate one token at a time
- Each token requires full model forward pass
- Cannot parallelize across sequence dimension during generation
- Memory bandwidth bound, not compute bound

**Why It's Slow**:
```
For each token:
1. Load all model parameters from memory
2. Compute one output token
3. Repeat

Problem: Loading 70B parameters (140GB) to generate 1 token (few bytes)
Inefficiency: ~99.999% of data movement is parameter loading!
```

**Attempted Solutions (2020-2022)**:
- Faster hardware (helps but expensive)
- Quantization (reduces parameter size)
- Distillation (smaller models)
- None broke the sequential bottleneck

In [ ]:
def analyze_generation_cost(model_params_gb, tokens_to_generate, memory_bandwidth_gb_s):
    """
    Analyze the cost of autoregressive generation.
    
    Args:
        model_params_gb: Model size in GB
        tokens_to_generate: Number of tokens to generate
        memory_bandwidth_gb_s: Memory bandwidth in GB/s
    """
    # Time to load model parameters once
    time_per_load = model_params_gb / memory_bandwidth_gb_s
    
    # Total time (load params for each token)
    total_time = time_per_load * tokens_to_generate
    
    # Tokens per second
    tokens_per_sec = tokens_to_generate / total_time
    
    return {
        'time_per_token_ms': time_per_load * 1000,
        'total_time_s': total_time,
        'tokens_per_sec': tokens_per_sec,
        'throughput_tokens_per_sec': tokens_per_sec
    }

# Compare different model sizes
configs = [
    ('Llama 2 7B', 14),    # fp16
    ('Llama 2 13B', 26),
    ('Llama 2 70B', 140),
    ('GPT-3 175B', 350),
]

# Typical hardware specs
hardware = [
    ('A100 80GB', 2000),      # GB/s
    ('H100 80GB', 3350),      # GB/s
    ('4x A100 (TP)', 8000),   # Aggregated
]

print("Autoregressive Generation Analysis")
print("=" * 90)

for hw_name, bandwidth in hardware:
    print(f"\n{hw_name} (Memory Bandwidth: {bandwidth} GB/s):")
    print(f"  {'Model':<15} {'Time/Token (ms)':<18} {'Tokens/sec':<15} {'100 tokens (s)'}")
    print("  " + "-" * 70)
    
    for model_name, model_size_gb in configs:
        result = analyze_generation_cost(model_size_gb, 100, bandwidth)
        print(f"  {model_name:<15} {result['time_per_token_ms']:<18.1f} "
              f"{result['tokens_per_sec']:<15.1f} {result['total_time_s']:.2f}")

In [ ]:
# Visualize the bottleneck
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model size vs throughput
model_sizes = [14, 26, 70, 140, 350]
model_names = ['7B', '13B', '70B', '140B', '350B']
bandwidth = 2000  # A100

throughputs = [bandwidth / size for size in model_sizes]

axes[0].bar(model_names, throughputs, color='#e74c3c', alpha=0.7)
axes[0].set_ylabel('Tokens/Second', fontsize=12)
axes[0].set_xlabel('Model Size', fontsize=12)
axes[0].set_title('Throughput vs Model Size (A100 80GB)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Time breakdown for one token
categories = ['Load\nParameters', 'Compute', 'Memory\nWrite']
times = [70, 1, 0.1]  # Llama 2 70B on A100 (ms)
colors_breakdown = ['#e74c3c', '#3498db', '#2ecc71']

axes[1].bar(categories, times, color=colors_breakdown, alpha=0.7)
axes[1].set_ylabel('Time (ms)', fontsize=12)
axes[1].set_title('Time Breakdown per Token (Llama 2 70B)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_yscale('log')

# Add percentage labels
total_time = sum(times)
for i, (cat, time) in enumerate(zip(categories, times)):
    pct = time / total_time * 100
    axes[1].text(i, time, f'{pct:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("1. Throughput inversely proportional to model size")
print("2. Parameter loading dominates time (98%+)")
print("3. Memory bandwidth bound, not compute bound")
print("4. Need to amortize parameter loads across multiple tokens")

## 2. Speculative Decoding Concept

### Historical Context

**Key Papers**:
- "Fast Inference from Transformers via Speculative Decoding" (Leviathan et al., DeepMind, 2023)
- "Accelerating Large Language Model Decoding with Speculative Sampling" (Chen et al., Google, 2023)

**Core Insight**:
```
Problem: Must generate tokens sequentially
Observation: Small models can draft multiple tokens quickly
Solution: Generate K draft tokens, verify them in parallel
```

**Algorithm**:
```python
1. Draft Phase:
   - Use small, fast draft model M_draft
   - Generate K candidate tokens autoregressively
   - Cost: K × (small model forward pass)

2. Verification Phase:
   - Use large target model M_target
   - Process all K draft tokens in PARALLEL
   - Cost: 1 × (large model forward pass)

3. Acceptance:
   - Compare draft and target distributions
   - Accept tokens probabilistically
   - If rejected: sample from corrected distribution
   - Guaranteed to match target model distribution!
```

**Why It Works**:
- Draft model is 10-100x faster (smaller)
- Target model verifies K tokens in one pass (parallel)
- If draft is often correct, big speedup
- If draft is wrong, no quality loss (rejection sampling)

**Key Guarantee**:
Output distribution is **identical** to standard sampling from target model!

In [ ]:
# Visualize speculative decoding
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Standard autoregressive decoding
ax = axes[0]
steps = 8
for i in range(steps):
    # Each token requires full model pass
    rect = plt.Rectangle((i, 0), 0.8, 1, color='#e74c3c', alpha=0.7, edgecolor='black')
    ax.add_patch(rect)
    ax.text(i + 0.4, 0.5, f'T{i+1}', ha='center', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(-0.5, steps)
ax.set_ylim(-0.2, 1.5)
ax.set_ylabel('Model Pass', fontsize=12)
ax.set_title('Standard Autoregressive: 8 tokens = 8 large model passes', 
             fontsize=13, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([0.5], ['Large Model'])
ax.text(steps + 0.5, 0.5, '→ SLOW', fontsize=12, fontweight='bold', color='red')

# Speculative decoding
ax = axes[1]

# Iteration 1: Draft 4 tokens, verify
draft_color = '#3498db'
verify_color = '#2ecc71'
reject_color = '#e67e22'

# Draft phase
for i in range(4):
    rect = plt.Rectangle((i * 0.3, 0), 0.25, 0.4, color=draft_color, alpha=0.7, edgecolor='black')
    ax.add_patch(rect)
    ax.text(i * 0.3 + 0.125, 0.2, f'd{i+1}', ha='center', va='center', fontsize=8)

# Verification (parallel)
rect = plt.Rectangle((1.4, 0.5), 0.8, 0.4, color=verify_color, alpha=0.7, edgecolor='black')
ax.add_patch(rect)
ax.text(1.8, 0.7, 'Verify\n(parallel)', ha='center', va='center', fontsize=8)

# Accept 3, reject 1
for i in range(3):
    ax.text(i * 0.3 + 0.125, -0.2, '✓', ha='center', va='center', 
            fontsize=14, color='green', fontweight='bold')
ax.text(0.9 + 0.125, -0.2, '✗', ha='center', va='center', 
        fontsize=14, color='red', fontweight='bold')

# Iteration 2
offset = 2.5
for i in range(4):
    rect = plt.Rectangle((offset + i * 0.3, 0), 0.25, 0.4, color=draft_color, alpha=0.7, edgecolor='black')
    ax.add_patch(rect)
    ax.text(offset + i * 0.3 + 0.125, 0.2, f'd{i+5}', ha='center', va='center', fontsize=8)

rect = plt.Rectangle((offset + 1.4, 0.5), 0.8, 0.4, color=verify_color, alpha=0.7, edgecolor='black')
ax.add_patch(rect)
ax.text(offset + 1.8, 0.7, 'Verify\n(parallel)', ha='center', va='center', fontsize=8)

for i in range(4):
    ax.text(offset + i * 0.3 + 0.125, -0.2, '✓', ha='center', va='center', 
            fontsize=14, color='green', fontweight='bold')

ax.set_xlim(-0.5, 6)
ax.set_ylim(-0.5, 1.2)
ax.set_ylabel('Phase', fontsize=12)
ax.set_title('Speculative Decoding: 8 tokens = 2 large model passes (4x speedup)', 
             fontsize=13, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([0.2, 0.7], ['Draft (fast)', 'Verify (parallel)'])
ax.text(6.2, 0.5, '→ FAST', fontsize=12, fontweight='bold', color='green')

plt.tight_layout()
plt.show()

print("Key Advantages:")
print("1. Draft model is 10-100x faster (much smaller)")
print("2. Target model verifies K tokens in parallel (1 pass)")
print("3. Accepted tokens get 'for free' (no extra large model pass)")
print("4. Quality is IDENTICAL to target model (rejection sampling guarantee)")

## 3. Speculative Decoding Algorithm

In [ ]:
def speculative_sample(
    draft_probs: torch.Tensor,
    target_probs: torch.Tensor,
    draft_token: int,
) -> Tuple[int, bool]:
    """
    Speculative sampling with rejection.
    
    Key insight: Accept draft token with probability min(1, p_target / p_draft)
    If rejected, sample from adjusted distribution.
    
    This guarantees the output distribution matches target model!
    
    Args:
        draft_probs: Draft model distribution [vocab_size]
        target_probs: Target model distribution [vocab_size]
        draft_token: Token chosen by draft model
    
    Returns:
        (token, accepted): Final token and whether draft was accepted
    """
    # Acceptance probability
    r = target_probs[draft_token] / draft_probs[draft_token]
    
    # Accept with probability min(1, r)
    if torch.rand(1).item() < min(1.0, r.item()):
        return draft_token, True
    
    # Rejection: sample from adjusted distribution
    # p_adjusted = max(0, p_target - p_draft) / sum(max(0, p_target - p_draft))
    adjusted_probs = torch.clamp(target_probs - draft_probs, min=0)
    adjusted_probs = adjusted_probs / adjusted_probs.sum()
    
    new_token = torch.multinomial(adjusted_probs, num_samples=1).item()
    return new_token, False


class SpeculativeDecoder:
    """
    Speculative decoding implementation.
    
    Combines small draft model with large target model for faster generation.
    """
    def __init__(
        self,
        draft_model: Callable,
        target_model: Callable,
        k: int = 4,  # Number of speculative tokens
        temperature: float = 1.0,
    ):
        self.draft_model = draft_model
        self.target_model = target_model
        self.k = k
        self.temperature = temperature
        
        # Statistics
        self.stats = {
            'total_tokens': 0,
            'accepted_tokens': 0,
            'target_calls': 0,
            'draft_calls': 0,
        }
    
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int,
    ) -> torch.Tensor:
        """
        Generate tokens using speculative decoding.
        
        Args:
            input_ids: Input token IDs [batch_size, seq_len]
            max_new_tokens: Maximum number of tokens to generate
        
        Returns:
            Generated token IDs [batch_size, seq_len + max_new_tokens]
        """
        current_ids = input_ids.clone()
        tokens_generated = 0
        
        while tokens_generated < max_new_tokens:
            # Phase 1: Draft K tokens with draft model
            draft_tokens = []
            draft_probs_list = []
            draft_input = current_ids.clone()
            
            for _ in range(self.k):
                logits = self.draft_model(draft_input)
                probs = F.softmax(logits[:, -1] / self.temperature, dim=-1)
                token = torch.multinomial(probs, num_samples=1)
                
                draft_tokens.append(token.item())
                draft_probs_list.append(probs[0])
                
                draft_input = torch.cat([draft_input, token], dim=-1)
                self.stats['draft_calls'] += 1
            
            # Phase 2: Verify with target model (parallel)
            # Process all K draft tokens in one forward pass
            verify_input = torch.cat([
                current_ids,
                torch.tensor([draft_tokens], device=current_ids.device)
            ], dim=-1)
            
            target_logits = self.target_model(verify_input)
            self.stats['target_calls'] += 1
            
            # Get target probabilities for each position
            target_probs_list = []
            for i in range(len(draft_tokens)):
                pos = -(len(draft_tokens) - i + 1)
                probs = F.softmax(target_logits[:, pos] / self.temperature, dim=-1)
                target_probs_list.append(probs[0])
            
            # Phase 3: Accept/Reject
            accepted = []
            for i, (draft_token, draft_probs, target_probs) in enumerate(
                zip(draft_tokens, draft_probs_list, target_probs_list)
            ):
                token, is_accepted = speculative_sample(
                    draft_probs, target_probs, draft_token
                )
                accepted.append(token)
                
                self.stats['total_tokens'] += 1
                if is_accepted:
                    self.stats['accepted_tokens'] += 1
                
                if not is_accepted:
                    # Stop at first rejection
                    break
            
            # Add accepted tokens
            accepted_tensor = torch.tensor([accepted], device=current_ids.device)
            current_ids = torch.cat([current_ids, accepted_tensor], dim=-1)
            tokens_generated += len(accepted)
            
            if tokens_generated >= max_new_tokens:
                break
        
        return current_ids[:, :input_ids.shape[1] + max_new_tokens]
    
    def get_stats(self):
        """Get performance statistics."""
        acceptance_rate = self.stats['accepted_tokens'] / max(self.stats['total_tokens'], 1)
        avg_accepted = self.stats['accepted_tokens'] / max(self.stats['target_calls'], 1)
        
        return {
            **self.stats,
            'acceptance_rate': acceptance_rate,
            'avg_accepted_per_call': avg_accepted,
        }

print("Speculative Decoding Algorithm Implemented")
print("="*60)
print("\nKey Components:")
print("1. speculative_sample(): Rejection sampling for quality guarantee")
print("2. Draft phase: Generate K candidates quickly")
print("3. Verify phase: Process all K tokens in parallel")
print("4. Accept/Reject: Keep accepted tokens, resample rejected")
print("\nGuarantee: Output distribution IDENTICAL to target model")

## 4. Mock Implementation and Testing

In [ ]:
class MockLanguageModel:
    """
    Mock language model for testing speculative decoding.
    """
    def __init__(self, vocab_size=1000, latency_ms=1.0, quality=1.0):
        self.vocab_size = vocab_size
        self.latency_ms = latency_ms
        self.quality = quality  # How similar to target (0-1)
        
    def __call__(self, input_ids):
        """
        Generate logits for next token.
        
        Quality parameter controls how similar draft is to target:
        - quality=1.0: Identical distribution
        - quality=0.0: Random distribution
        """
        # Simulate latency
        time.sleep(self.latency_ms / 1000)
        
        batch_size, seq_len = input_ids.shape
        
        # Generate logits based on last token (simple pattern)
        last_token = input_ids[:, -1]
        
        # Base logits follow a pattern
        logits = torch.randn(batch_size, seq_len, self.vocab_size)
        
        # Make certain tokens more likely based on context
        next_token_bias = (last_token + 1) % self.vocab_size
        logits[:, -1, next_token_bias] += 2.0 * self.quality
        
        # Add some noise (less for higher quality)
        noise = torch.randn_like(logits) * (1 - self.quality)
        logits = logits + noise
        
        return logits

# Create draft and target models
draft_model = MockLanguageModel(
    vocab_size=1000,
    latency_ms=1.0,   # 10x faster than target
    quality=0.7       # 70% similar to target
)

target_model = MockLanguageModel(
    vocab_size=1000,
    latency_ms=10.0,  # Large model
    quality=1.0       # Ground truth
)

# Test speculative decoding
print("Testing Speculative Decoding")
print("="*70)

input_ids = torch.randint(0, 1000, (1, 10))
max_new_tokens = 50

# Compare regular vs speculative
print("\n1. Regular Autoregressive Generation:")
start = time.time()
regular_output = input_ids.clone()
for _ in range(max_new_tokens):
    logits = target_model(regular_output)
    probs = F.softmax(logits[:, -1], dim=-1)
    next_token = torch.multinomial(probs, num_samples=1)
    regular_output = torch.cat([regular_output, next_token], dim=-1)
regular_time = time.time() - start
print(f"   Time: {regular_time:.2f}s")
print(f"   Tokens/sec: {max_new_tokens / regular_time:.1f}")

# Speculative decoding
print("\n2. Speculative Decoding (k=4):")
decoder = SpeculativeDecoder(
    draft_model=draft_model,
    target_model=target_model,
    k=4,
    temperature=1.0,
)

start = time.time()
spec_output = decoder.generate(input_ids, max_new_tokens)
spec_time = time.time() - start

stats = decoder.get_stats()
print(f"   Time: {spec_time:.2f}s")
print(f"   Tokens/sec: {max_new_tokens / spec_time:.1f}")
print(f"   Speedup: {regular_time / spec_time:.2f}x")
print(f"\n   Statistics:")
print(f"     Target model calls: {stats['target_calls']}")
print(f"     Draft model calls: {stats['draft_calls']}")
print(f"     Acceptance rate: {stats['acceptance_rate']:.1%}")
print(f"     Avg tokens accepted per verification: {stats['avg_accepted_per_call']:.2f}")

## 5. Speedup Analysis

In [ ]:
def theoretical_speedup(k, alpha, r):
    """
    Calculate theoretical speedup of speculative decoding.
    
    Args:
        k: Number of draft tokens per iteration
        alpha: Acceptance rate (0-1)
        r: Speed ratio draft/target (e.g., 10 if draft is 10x faster)
    
    Formula:
        E[tokens per iteration] = k * alpha + 1  (accepted + correction)
        Cost per iteration = k/r + 1  (draft cost + verify cost)
        Speedup = E[tokens] / Cost
    """
    expected_tokens = k * alpha + (1 - alpha ** k)  # More precise formula
    cost = k / r + 1
    speedup = expected_tokens / cost
    return speedup

# Analyze speedup for different parameters
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Speedup vs acceptance rate
ax = axes[0, 0]
acceptance_rates = np.linspace(0.3, 0.95, 50)
for k in [2, 4, 6, 8]:
    speedups = [theoretical_speedup(k, alpha, r=10) for alpha in acceptance_rates]
    ax.plot(acceptance_rates, speedups, label=f'k={k}', linewidth=2)

ax.set_xlabel('Acceptance Rate', fontsize=12)
ax.set_ylabel('Speedup', fontsize=12)
ax.set_title('Speedup vs Acceptance Rate (r=10)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Speedup vs k
ax = axes[0, 1]
k_values = np.arange(1, 17)
for alpha in [0.5, 0.6, 0.7, 0.8, 0.9]:
    speedups = [theoretical_speedup(k, alpha, r=10) for k in k_values]
    ax.plot(k_values, speedups, label=f'α={alpha:.1f}', linewidth=2)

ax.set_xlabel('Draft Length (k)', fontsize=12)
ax.set_ylabel('Speedup', fontsize=12)
ax.set_title('Speedup vs Draft Length (r=10)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Speedup vs speed ratio
ax = axes[1, 0]
speed_ratios = np.linspace(1, 50, 50)
for alpha in [0.5, 0.6, 0.7, 0.8, 0.9]:
    speedups = [theoretical_speedup(4, alpha, r) for r in speed_ratios]
    ax.plot(speed_ratios, speedups, label=f'α={alpha:.1f}', linewidth=2)

ax.set_xlabel('Speed Ratio (draft/target)', fontsize=12)
ax.set_ylabel('Speedup', fontsize=12)
ax.set_title('Speedup vs Speed Ratio (k=4)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Heatmap of speedup
ax = axes[1, 1]
k_range = np.arange(2, 11)
alpha_range = np.linspace(0.4, 0.95, 20)
speedup_matrix = np.zeros((len(alpha_range), len(k_range)))

for i, alpha in enumerate(alpha_range):
    for j, k in enumerate(k_range):
        speedup_matrix[i, j] = theoretical_speedup(k, alpha, r=10)

im = ax.imshow(speedup_matrix, aspect='auto', origin='lower', cmap='RdYlGn',
               extent=[k_range[0], k_range[-1], alpha_range[0], alpha_range[-1]])
ax.set_xlabel('Draft Length (k)', fontsize=12)
ax.set_ylabel('Acceptance Rate', fontsize=12)
ax.set_title('Speedup Heatmap (r=10)', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Speedup')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("1. Speedup increases with acceptance rate (more drafts accepted)")
print("2. Optimal k depends on acceptance rate (higher α → larger k)")
print("3. Speedup saturates as speed ratio increases")
print("4. Typical speedup range: 2-3x for realistic parameters")

In [ ]:
# Calculate optimal parameters for real models
print("Optimal Parameters for Real Models")
print("="*70)

scenarios = [
    {
        'name': 'Llama 2 70B + Llama 2 7B',
        'target_size': '70B',
        'draft_size': '7B',
        'speed_ratio': 10,
        'acceptance_rate': 0.7,
    },
    {
        'name': 'Llama 2 13B + Llama 2 7B',
        'target_size': '13B',
        'draft_size': '7B',
        'speed_ratio': 2,
        'acceptance_rate': 0.75,
    },
    {
        'name': 'GPT-3.5 + Distilled GPT',
        'target_size': '175B',
        'draft_size': '~6B',
        'speed_ratio': 30,
        'acceptance_rate': 0.65,
    },
]

for scenario in scenarios:
    print(f"\n{scenario['name']}:")
    print(f"  Target: {scenario['target_size']}, Draft: {scenario['draft_size']}")
    print(f"  Speed ratio: {scenario['speed_ratio']}x")
    print(f"  Acceptance rate: {scenario['acceptance_rate']:.0%}")
    print(f"\n  Optimal k values:")
    
    best_k = None
    best_speedup = 0
    
    for k in range(2, 13):
        speedup = theoretical_speedup(
            k, scenario['acceptance_rate'], scenario['speed_ratio']
        )
        print(f"    k={k:2d}: {speedup:.2f}x speedup")
        
        if speedup > best_speedup:
            best_speedup = speedup
            best_k = k
    
    print(f"\n  Recommended: k={best_k} ({best_speedup:.2f}x speedup)")

## 6. Choosing Draft Models

Critical decision: Which model to use for drafting?

In [ ]:
print("Draft Model Selection Guide")
print("="*80)

draft_options = [
    {
        'type': 'Smaller Model (Same Family)',
        'examples': 'Llama 2 7B for Llama 2 70B',
        'speed_ratio': '8-10x',
        'acceptance_rate': '65-75%',
        'pros': [
            'Same architecture and training',
            'High acceptance rate',
            'Easy to deploy (same codebase)',
            'Predictable quality'
        ],
        'cons': [
            'Still requires significant memory',
            'May not be available for all models'
        ],
        'recommended': 'YES - Best default choice'
    },
    {
        'type': 'Distilled Model',
        'examples': 'DistilGPT for GPT, TinyLlama for Llama',
        'speed_ratio': '20-50x',
        'acceptance_rate': '55-65%',
        'pros': [
            'Very fast',
            'Minimal memory overhead',
            'Trained to mimic target',
            'Can fit on CPU'
        ],
        'cons': [
            'Lower acceptance rate',
            'Requires distillation training',
            'May not generalize well'
        ],
        'recommended': 'For extreme memory constraints'
    },
    {
        'type': 'N-gram Model',
        'examples': 'Lookup table, KV cache',
        'speed_ratio': '100-1000x',
        'acceptance_rate': '30-40%',
        'pros': [
            'Extremely fast',
            'No additional model needed',
            'Works for repetitive text',
            'Trivial to implement'
        ],
        'cons': [
            'Very low acceptance rate',
            'Only works for predictable patterns',
            'Not useful for creative tasks'
        ],
        'recommended': 'For structured/repetitive outputs only'
    },
    {
        'type': 'Early Exit (Same Model)',
        'examples': 'Exit at layer 8 of 32-layer model',
        'speed_ratio': '3-4x',
        'acceptance_rate': '75-85%',
        'pros': [
            'No separate model needed',
            'High acceptance rate',
            'Shares KV cache with target',
            'Easy to implement'
        ],
        'cons': [
            'Limited speedup (still loading full model)',
            'Requires architectural support',
            'May need retraining'
        ],
        'recommended': 'For single-model deployment'
    },
]

for i, option in enumerate(draft_options, 1):
    print(f"\n{i}. {option['type']}")
    print(f"   Examples: {option['examples']}")
    print(f"   Speed ratio: {option['speed_ratio']}")
    print(f"   Acceptance rate: {option['acceptance_rate']}")
    print(f"   Recommendation: {option['recommended']}")
    print(f"\n   Pros:")
    for pro in option['pros']:
        print(f"     + {pro}")
    print(f"   Cons:")
    for con in option['cons']:
        print(f"     - {con}")

print("\n" + "="*80)
print("\nDecision Framework:")
print("""
1. Default: Use smaller model from same family
   - Llama 2 70B → Llama 2 7B
   - GPT-4 → GPT-3.5
   - Mistral 7B → TinyLlama 1B

2. Memory constrained: Use distilled model or n-gram
   - Deploy draft on CPU, target on GPU
   - Use smaller batch size for draft

3. Maximize speedup: Find optimal size/quality tradeoff
   - Benchmark acceptance rates with different draft models
   - Choose model that maximizes: speedup = (acceptance * speed_ratio)

4. Domain specific: Fine-tune draft model
   - Train draft to mimic target on your specific data
   - Can significantly improve acceptance rate
""")

## 7. Quality Preservation Analysis

In [ ]:
def verify_distribution_match(n_samples=10000, vocab_size=100):
    """
    Verify that speculative sampling produces identical distribution to target.
    """
    # Create mock distributions
    target_logits = torch.randn(vocab_size)
    target_probs = F.softmax(target_logits, dim=0)
    
    draft_logits = target_logits + torch.randn(vocab_size) * 0.5  # Somewhat similar
    draft_probs = F.softmax(draft_logits, dim=0)
    
    # Sample from target directly
    target_samples = torch.multinomial(target_probs, num_samples=n_samples, replacement=True)
    target_counts = torch.bincount(target_samples, minlength=vocab_size).float()
    target_dist = target_counts / target_counts.sum()
    
    # Sample using speculative sampling
    spec_samples = []
    for _ in range(n_samples):
        draft_token = torch.multinomial(draft_probs, num_samples=1).item()
        token, _ = speculative_sample(draft_probs, target_probs, draft_token)
        spec_samples.append(token)
    
    spec_samples = torch.tensor(spec_samples)
    spec_counts = torch.bincount(spec_samples, minlength=vocab_size).float()
    spec_dist = spec_counts / spec_counts.sum()
    
    return target_dist, spec_dist, target_probs

# Verify distribution match
target_dist, spec_dist, target_probs = verify_distribution_match()

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Top 20 tokens
top_k = 20
top_indices = torch.argsort(target_probs, descending=True)[:top_k]

ax = axes[0]
x = np.arange(top_k)
width = 0.35
ax.bar(x - width/2, target_dist[top_indices], width, label='Target (direct)', alpha=0.7)
ax.bar(x + width/2, spec_dist[top_indices], width, label='Speculative', alpha=0.7)
ax.set_xlabel('Token Rank', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Distribution Comparison (Top 20 Tokens)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Scatter plot
ax = axes[1]
ax.scatter(target_dist, spec_dist, alpha=0.5)
ax.plot([0, target_dist.max()], [0, target_dist.max()], 'r--', label='Perfect match')
ax.set_xlabel('Target Distribution', fontsize=12)
ax.set_ylabel('Speculative Distribution', fontsize=12)
ax.set_title('Distribution Match (All Tokens)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# KL divergence
ax = axes[2]
kl_div = F.kl_div(spec_dist.log(), target_dist, reduction='batchmean').item()
js_div = 0.5 * (F.kl_div(spec_dist.log(), (spec_dist + target_dist)/2, reduction='batchmean') +
                F.kl_div(target_dist.log(), (spec_dist + target_dist)/2, reduction='batchmean')).item()

metrics = ['KL Divergence', 'JS Divergence', 'Theoretical\nLimit']
values = [kl_div * 1000, js_div * 1000, 0]  # Scale for visibility
colors_metrics = ['#e74c3c', '#3498db', '#2ecc71']

bars = ax.bar(metrics, values, color=colors_metrics, alpha=0.7)
ax.set_ylabel('Divergence (×10⁻³)', fontsize=12)
ax.set_title('Distribution Divergence', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nQuality Preservation Verification")
print("="*70)
print(f"KL Divergence: {kl_div:.6f} (should be ~0)")
print(f"JS Divergence: {js_div:.6f} (should be ~0)")
print(f"Max absolute difference: {(target_dist - spec_dist).abs().max():.6f}")
print("\nConclusion: Distributions are IDENTICAL (within sampling error)")
print("Speculative decoding provides mathematical guarantee of quality preservation!")

## 8. Production Deployment

### Integration with Existing Frameworks

In [ ]:
print("Production Deployment Guide")
print("="*80)

deployment_guide = """
1. FRAMEWORK SUPPORT (as of 2024):

   vLLM:
   ✓ Native speculative decoding support
   ✓ Combined with PagedAttention
   ✓ Automatic draft model loading
   
   from vllm import LLM, SamplingParams
   
   llm = LLM(
       model="meta-llama/Llama-2-70b-hf",
       speculative_model="meta-llama/Llama-2-7b-hf",
       num_speculative_tokens=4,
       use_v2_block_manager=True,
   )

   Hugging Face Transformers:
   ✓ Assisted generation (similar concept)
   ✓ Works with any model
   
   from transformers import AutoModelForCausalLM
   
   model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-70b-hf")
   assistant = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
   
   outputs = model.generate(
       input_ids,
       assistant_model=assistant,
       do_sample=True,
   )

   TensorRT-LLM:
   ✓ Speculative sampling in latest versions
   ✓ Optimized for NVIDIA GPUs
   ✓ Can combine with quantization


2. DEPLOYMENT ARCHITECTURES:

   A. Co-located (Same GPU):
   - Both models on same GPU
   - Best for memory-efficient draft models
   - Simpler deployment
   
   Pros: Lower latency, no network overhead
   Cons: Memory pressure, may limit batch size

   B. Split (Draft on CPU, Target on GPU):
   - Draft model on CPU
   - Target model on GPU
   - Maximizes GPU memory for target
   
   Pros: Full GPU for target, larger batches
   Cons: CPU-GPU transfer overhead

   C. Multi-GPU:
   - Draft on separate GPU
   - Target on multiple GPUs (tensor parallel)
   - Best for very large models
   
   Pros: Maximum throughput, no memory trade-off
   Cons: Higher hardware cost, GPU-GPU transfer


3. CONFIGURATION TUNING:

   Key Parameters:
   - num_speculative_tokens (k): Start with 4-6
   - acceptance_threshold: Usually don't set (use rejection sampling)
   - max_draft_length: Cap to prevent runaway drafting
   
   Tuning Process:
   1. Measure baseline throughput (no speculation)
   2. Enable speculation with k=4
   3. Monitor acceptance rate
   4. If acceptance > 70%: increase k
   5. If acceptance < 50%: decrease k or change draft model
   6. Benchmark end-to-end throughput


4. MONITORING:

   Key Metrics:
   - Acceptance rate (target: 60-80%)
   - Average accepted tokens per verification
   - End-to-end latency
   - Throughput (tokens/sec)
   - GPU memory usage
   
   Red Flags:
   - Acceptance rate < 40%: Draft model too different
   - Acceptance rate > 95%: Can increase k
   - Throughput decrease: Overhead too high, reduce k


5. OPTIMIZATION CHECKLIST:

   ✓ Choose draft model from same family
   ✓ Benchmark multiple k values (typically 4-8)
   ✓ Combine with other optimizations (PagedAttention, quantization)
   ✓ Monitor acceptance rates in production
   ✓ A/B test against standard decoding
   ✓ Consider domain-specific fine-tuning of draft
   ✓ Profile memory usage (draft + target)
   ✓ Test with realistic sequence lengths
   ✓ Validate output quality on benchmarks


6. COMMON PITFALLS:

   ✗ Using unrelated draft model (low acceptance)
   ✗ Setting k too large (diminishing returns)
   ✗ Not monitoring acceptance rates
   ✗ Assuming speedup matches theory (always benchmark!)
   ✗ Forgetting draft model memory overhead
   ✗ Using on short generations (overhead dominates)
   ✗ Not combining with other optimizations
"""

print(deployment_guide)

## 9. When to Use Speculative Decoding

In [ ]:
import pandas as pd

# Decision matrix
scenarios = [
    {
        'Use Case': 'Long-form generation (>100 tokens)',
        'Recommended': 'YES',
        'Expected Speedup': '2-3x',
        'Reasoning': 'Amortizes overhead, high token count',
        'Best Config': 'k=4-6, same-family draft'
    },
    {
        'Use Case': 'Short responses (<20 tokens)',
        'Recommended': 'NO',
        'Expected Speedup': '0.8-1.2x',
        'Reasoning': 'Overhead dominates, few tokens to save',
        'Best Config': 'Use standard decoding'
    },
    {
        'Use Case': 'Code generation',
        'Recommended': 'YES',
        'Expected Speedup': '2.5-4x',
        'Reasoning': 'High predictability, structured output',
        'Best Config': 'k=6-8, code-specific draft'
    },
    {
        'Use Case': 'Creative writing',
        'Recommended': 'MAYBE',
        'Expected Speedup': '1.5-2x',
        'Reasoning': 'Lower acceptance rate (less predictable)',
        'Best Config': 'k=4, high-quality draft'
    },
    {
        'Use Case': 'Structured data (JSON, SQL)',
        'Recommended': 'YES',
        'Expected Speedup': '3-5x',
        'Reasoning': 'Very high predictability, format constraints',
        'Best Config': 'k=8-12, grammar-constrained'
    },
    {
        'Use Case': 'Low-latency serving (<50ms)',
        'Recommended': 'NO',
        'Expected Speedup': '0.9-1.1x',
        'Reasoning': 'Added complexity, variable latency',
        'Best Config': 'Use quantization + small model'
    },
    {
        'Use Case': 'Batch inference (large batches)',
        'Recommended': 'MAYBE',
        'Expected Speedup': '1.2-1.8x',
        'Reasoning': 'Already efficient, memory pressure',
        'Best Config': 'Profile first, k=2-4 if beneficial'
    },
    {
        'Use Case': 'Memory-constrained (<24GB)',
        'Recommended': 'MAYBE',
        'Expected Speedup': '1.5-2x',
        'Reasoning': 'Draft model adds memory, but CPU offload possible',
        'Best Config': 'Tiny draft or CPU-based draft'
    },
]

df = pd.DataFrame(scenarios)

print("\nSpeculative Decoding - Decision Matrix")
print("="*100)
print(df.to_string(index=False))

print("\n\nQuick Decision Guide:")
print("="*80)
print("""
USE Speculative Decoding when:
✓ Generating many tokens (>50)
✓ Output is somewhat predictable
✓ Throughput is bottleneck (not latency)
✓ Have suitable draft model available
✓ Can afford extra memory/complexity

SKIP Speculative Decoding when:
✗ Generating few tokens (<20)
✗ Ultra-low latency required
✗ Memory extremely constrained
✗ No good draft model available
✗ Already using large batch sizes efficiently

TYPICAL BEST CASES:
1. Code generation: 3-4x speedup
2. Long-form content: 2-3x speedup
3. Structured output: 3-5x speedup

TYPICAL WORST CASES:
1. Short responses: 0.8-1.2x (overhead)
2. Highly creative: 1.2-1.5x (low acceptance)
3. Small batch: overhead dominates
""")

## Summary

### Key Takeaways

1. **The Problem**:
   - Autoregressive generation is memory-bandwidth bound
   - Must load full model parameters for each token
   - Cannot parallelize across sequence dimension

2. **The Solution**:
   - Use fast draft model to generate K candidate tokens
   - Verify all K tokens in parallel with target model
   - Accept tokens probabilistically via rejection sampling
   - **Guarantee**: Output distribution identical to target!

3. **Performance**:
   - Typical speedup: 2-3x
   - Best case (structured output): 3-5x
   - Depends on: acceptance rate, draft speed, k value
   - Formula: speedup ≈ (α × k) / (k/r + 1)

4. **Draft Model Selection**:
   - **Best**: Smaller model from same family (e.g., 7B for 70B)
   - **Alternative**: Distilled model (higher speed, lower acceptance)
   - **Experimental**: Early exit, n-gram models
   - Acceptance rate target: 60-80%

5. **Production Deployment**:
   - Supported in vLLM, Hugging Face, TensorRT-LLM
   - Start with k=4-6
   - Monitor acceptance rates
   - Combine with PagedAttention and quantization
   - Best for long-form, code, and structured generation

6. **Quality Guarantee**:
   - Mathematical proof: output distribution identical to target
   - No approximation or quality loss
   - Rejection sampling ensures correctness
   - This is the key advantage over other speedup techniques!

### Historical Impact

- **2023**: Speculative decoding papers (DeepMind, Google)
- **2023-2024**: Rapid adoption in all major frameworks
- **2024**: Standard technique for production LLM serving
- **Impact**: Enabled 2-3x throughput increase with zero quality loss

### Comparison to Other Techniques

| Technique | Speedup | Quality | Complexity |
|-----------|---------|---------|------------|
| Speculative Decoding | 2-3x | Identical | Medium |
| Quantization | 1.5-2x | Slight loss | Low |
| Distillation | 5-10x | Noticeable loss | High |
| PagedAttention | 1.5x | No change | Low |
| **Combined** | **5-10x** | **Minimal loss** | **Medium** |

### Next Steps

- **Notebook 57**: vLLM Serving (PagedAttention + Speculative Decoding)
- **Notebook 58**: TensorRT-LLM (NVIDIA optimizations)
- **Notebook 59**: Continuous Batching (maximize throughput)